# In this mini project we'll do option pricing using monte carlo and then accelerate it with CUDA

In [4]:
import numpy as np
import time
import math
from numba import cuda
from scipy.stats import norm


In [31]:
S0    = 100.0    # Initial stock price
K     = 105.0    # Strike price which is the price
r     = 0.05     # Risk-free interest rate which is the guaranteed ROI
sigma = 0.2      # Volatility in percentages
T     = 1.0      # Time to maturity in years
N     = 100_000_000  # Number of simulations

In [10]:
def monte_carlo_sequential(S0, K, r, sigma, T, N):
    payoffs = np.zeros(N)
    # This is the naive loop monte carlo sequential option pricing
    # It is very slow due to the absence of parallelism and vectorization
    for i in range(N):
        Z = np.random.standard_normal()
        # This is the geometric brownian motion
        ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
        payoffs[i] = max(ST - K, 0.0)

    option_price = np.exp(-r * T) * np.mean(payoffs)
    return option_price

In [30]:
# Perf counter and measure of time
start = time.perf_counter()
price = monte_carlo_sequential(S0, K, r, sigma, T, N)
end = time.perf_counter()
sequential_time = end - start
print(f"\n  Sequential Time  : {sequential_time:.4f} seconds")
print(price)

KeyboardInterrupt: 

In [26]:
def monte_carlo_option_cpu(S0, K, r, sigma, T, N):
    # This is the Numpy version which is lightning fast compared to the naive loop version

    Z = np.random.standard_normal(N)
    S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    payoffs = np.maximum(S_T - K, 0)
    price = np.exp(-r * T) * np.mean(payoffs)
    return price
# Why didn't we use loops here ?
# Because first by definition loops are very slow, and for monte carlo option pricing we consider that it is a european option which means that we only care about the price at maturity which means that N and N-1 are independent

In [28]:
start = time.perf_counter()
numpy_price = monte_carlo_option_cpu(S0, K, r, sigma, T, N)
end = time.perf_counter()
numpy_sequential_time = end - start
print(numpy_price)
print(f"\n  Sequential Time  : {numpy_sequential_time:.4f} seconds")

8.020792605817347

  Sequential Time  : 5.3314 seconds


In [34]:
@cuda.jit()
def monte_carlo_naive_kernel(S0, K, r, sigma, T, seeds, payoffs):
    i = cuda.grid(1)
    if i >= payoffs.shape[0]:
        return
    # Here each thread generates it's own random number
    seed = seeds[i]
    seed = (seed * 1664525 + 1013904223) & 0xFFFFFFFF
    u = seed / 0xFFFFFFFF

    seed2 = (seed * 1664525 + 1013904223) & 0xFFFFFFFF
    u2 = seed2 / 0xFFFFFFFF
    Z = math.sqrt(-2.0 * math.log(u + 1e-10)) * math.cos(2.0 * math.pi * u2)

    # Each thread computes its own S_T and payoff
    S_T = S0 * math.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * Z)
    payoffs[i] = max(S_T - K, 0.0)



In [35]:
def monte_carlo_naive_gpu(S0, K, r, sigma, T, N):
    # Prepare random seeds — one per thread
    seeds = np.random.randint(0, 2**31, size=N, dtype=np.int64)
    seeds_dev = cuda.to_device(seeds)
    payoffs_dev = cuda.device_array(N, dtype=np.float64)

    # Launch N threads
    threads_per_block = 256
    blocks = (N + threads_per_block - 1) // threads_per_block

    monte_carlo_naive_kernel[blocks, threads_per_block](
        S0, K, r, sigma, T, seeds_dev, payoffs_dev
    )
    cuda.synchronize()

    payoffs = payoffs_dev.copy_to_host()
    return np.exp(-r * T) * np.mean(payoffs)

    # Compute option price
    price = np.exp(-r * T) * np.mean(payoffs)
    return price

In [36]:
start = time.perf_counter()
price_GPU = monte_carlo_naive_gpu(S0, K, r, sigma, T, N)
end = time.perf_counter()

gpu_naive_time = end - start
print(f"\n  GPU Time  : {gpu_naive_time:.4f} seconds")
print(price_GPU)


NvvmSupportError: libNVVM cannot be found. Do `conda install cudatoolkit`:
Could not find module 'nvvm.dll' (or one of its dependencies). Try using the full path with constructor syntax.